# DS2002 · JSON and Nested Records

**Lecture — 2026-09-28 · Fall 2026**  
**Class time:** 45 minutes

---

## Today's investigation

A campus research team has been running air-quality sensors on grounds. They send you one JSON document: some campaign metadata, two stations, and four hourly readings. One of the readings is missing.

**The question: which station has the highest mean PM2.5?**

That sounds like a one-line `groupby`, and eventually it is. But the measurements are buried inside nested arrays, one observation is `null`, and the units live somewhere else entirely in the document. None of that fits in a table yet.

The data is fictional and deliberately tiny, so you can check every transformation by hand. The target is one row per station per hour.

By the end you should be able to:

1. Explain what JSON is and where it came from.
2. Read a nested structure and trace a path to any value inside it.
3. Tell apart *it parses*, *it matches the contract*, and *it is fit to analyze*.
4. Load, reshape, summarize, and export JSON with Python.

## What JSON is

JSON stands for JavaScript Object Notation. Ignore the JavaScript part for now.

It is a way of writing structured data as plain text, so one program can hand data to another program that knows nothing about it. That is the entire job. A JSON document is characters — you can open it in Notepad, paste it into an email, or print it to a screen.

Two words worth having:

- **Serializing** is writing a value out of memory into that text form.
- **Parsing** is reading the text back and rebuilding real values from it.

```json
{"station_id": "A1", "active": true}
```

That line is JSON text. After Python parses it you have a dictionary holding a string and a bool. They look almost identical printed on a page, and they are not the same thing. Keeping that straight is most of what trips people up early.

You have been looking at JSON all semester without being told: every `.ipynb` file is JSON. Your cells, your outputs, and the metadata that makes notebook merge conflicts so unpleasant in Git.

### Where it came from

| When | What happened | Why it matters |
|---|---|---|
| 1999 | ECMAScript, 3rd edition | Gave the world a compact notation for objects and arrays |
| Early 2000s | Douglas Crockford promotes JSON | Argued for a small format aimed at exchanging data |
| 2006 | RFC 4627 | A written description, and the `application/json` media type |
| 2013, then 2017 | ECMA-404, then RFC 8259 | Settled the syntax as language-independent |

The name is the only JavaScript left in it. You do not need a browser or a JavaScript engine to read or write JSON.

### Why people call it the language of the internet

Treat that phrase as a metaphor rather than a fact. Here is what actually comes back when you call an API:

```
HTTP/1.1 200 OK
Content-Type: application/json

{"station_id": "A1", "pm25": 8.1}
```

**HTTP carries the message. JSON represents the data inside it.** Two separate jobs. HTTP supplies the methods, headers, and status codes; JSON is just the body.

JSON won that slot for four unglamorous reasons: the syntax is small, the text is readable by a human, it can express nested records, and nearly every language turns it into native values with one function call.

### Where you will meet it

| Stage of the work | What arrives as JSON | The judgment it demands |
|---|---|---|
| Acquire | API responses, sensor events | What was collected, when, in what units? |
| Understand | Nested records and metadata | What does one row of my analysis represent? |
| Prepare | Missing fields, types that drift | Which records am I willing to accept? |
| Publish | Results for a dashboard or service | Can another program read what I wrote? |

JSON is usually called **semi-structured**, which does not mean structure-free. It means the structure is real but nobody enforces it: fields can vary between records, and the producer may change the shape in version 2 without telling you.

### The six value types

A complete JSON document holds exactly one root value, and that value is one of six kinds:

| JSON type | Example | What Python makes of it |
|---|---|---|
| object | `{"station_id": "A1"}` | `dict` |
| array | `[8.1, 12.4, null]` | `list` |
| string | `"A1"` | `str` |
| number | `8.1`  `9`  `-2`  `1.2e3` | `float` or `int` |
| boolean | `true`  `false` | `True` or `False` |
| null | `null` | `None` |

JSON itself does not separate integers from decimals — a number is just a number. Python's decoder makes that call for you, which is usually what you want and occasionally is not.

### Objects and arrays: names versus positions

The two containers do different jobs.

```json
{
  "station_id": "A1",
  "pm25": 8.1
}
```

An **object** identifies values by name. You reach `8.1` by asking for `"pm25"`.

```json
[
  {"hour": 9, "pm25": 8.1},
  {"hour": 10, "pm25": null}
]
```

An **array** identifies values by position. You reach the second reading with index `1`, because Python counts from zero.

One rule that catches people: **the order of an object's members means nothing.** A server may send the keys in a different order tomorrow and the document is unchanged. **Array order does mean something**, so do not reorder it casually.

### The syntax rules

```json
{
  "station_id": "A1",
  "active": true,
  "readings": [8.1, null]
}
```

- **Names and strings use double quotes.** Never single. Keys are strings too, so they get quotes as well.
- **A colon separates a name from its value**, and a comma separates one member or item from the next.
- **Braces and brackets match** at every level.
- **`true`, `false`, and `null` are lowercase.** Python's `True` and `None` are not JSON.
- **No comma after the last item.**
- **No comments.** There is nowhere to leave a note.
- **Whitespace between tokens does not matter**, so indenting is purely for the reader.

Break any one of these and the whole document fails to parse. JSON gives no partial credit.

### Strings and numbers: the edge cases

| Case | Valid JSON | What to watch |
|---|---|---|
| Quote inside a string | `"He said \\"yes\\""` | Escape it with a backslash |
| Line break | `"line 1\\nline 2"` | Use the escape; a real newline inside quotes is invalid |
| Number | `-0.25`  `1.2e3` | No `+8`, no `.25`, no `01`, no `NaN`, no `Infinity` |
| Identifier | `"00127"` | Keep it a string or the leading zeros vanish |
| Date or time | `"2026-09-27T09:00:00Z"` | Just a string until your code interprets it |

Two of those cost people real time. A ZIP code or a student ID stored as a number loses its leading zeros permanently. And **JSON has no date type** — a timestamp is text, so `"2026-09-28"` and `"09/28/2026"` are equally valid and equally meaningless to the parser. Converting them is your job, with the same `pd.to_datetime` from last week.

## The dataset

Here is the whole document: campaign metadata at the top, an array of stations, and inside each station an array of readings.

The next cell writes it to `campus_air.json` and reads it straight back, so you see both halves of the round trip.

In [ ]:
import json
from pathlib import Path

CAMPUS_AIR = '''
{
  "campaign": "grounds-air",
  "units": {"pm25": "µg/m³"},
  "stations": [
    {"station_id": "A1", "site": {"name": "North"},
     "readings": [
       {"hour": 9,  "pm25": 8.1},
       {"hour": 10, "pm25": null}]},
    {"station_id": "B2", "site": {"name": "South"},
     "readings": [
       {"hour": 9,  "pm25": 12.4},
       {"hour": 10, "pm25": 10.8}]}
  ]
}
'''

Path('campus_air.json').write_text(CAMPUS_AIR, encoding='utf-8')

with open('campus_air.json', encoding='utf-8') as f:
    data = json.load(f)

print(type(data).__name__)
print(data['campaign'])
print(len(data['stations']), 'stations')

`json.load` reads from an open file. `json.loads` — with the **s**, for string — reads text you already have in memory. Both hand back ordinary Python dictionaries and lists, all the way down. There is no second parsing step.

Use `encoding='utf-8'` when you open the file. The units field contains `µg/m³`, and on a machine that guesses a different encoding those characters come back mangled.

### Reading a path

To pull one value out of a nested document, work down one container at a time and say out loud what you are standing in. A string in brackets selects an object member. An integer selects an array position.

In [ ]:
print(type(data['stations']).__name__, '- the list of stations')
print(type(data['stations'][0]).__name__, '- the A1 record')
print(repr(data['stations'][0]['site']['name']), '- a site name')
print(repr(data['stations'][0]['readings'][1]['pm25']), '- A1 at hour 10')

print()
print('units:', data['units']['pm25'])

The fourth line lands on `None`, because that reading arrived as `null`. The conversion is automatic and you cannot switch it off.

A wrong key raises `KeyError` and a wrong index raises `IndexError`. Those are loud and easy to fix. The quiet problem is the next one.

### Missing, null, empty, and zero are four different things

| What you see | What it means here | How to test it |
|---|---|---|
| `{}` | The field is absent entirely | `'pm25' not in r` |
| `{"pm25": null}` | The measurement is unavailable | `r['pm25'] is None` |
| `{"pm25": 0}` | A real measurement that happened to be zero | `r['pm25'] == 0` |
| `{"readings": []}` | No readings were recorded | `len(r['readings']) == 0` |
| `{"label": ""}` | A field that is present and blank | `r['label'] == ''` |

Syntax cannot tell you which of these you have. Your contract decides. Here, `null` means the sensor did not report a value that hour.

Now the trap. Python treats `0`, `0.0`, `''`, `[]`, `{}`, `None`, and `False` as false, so a casual `if value:` throws away real zeros along with the missing ones. On air quality data, a zero is a measurement somebody cares about.

In [ ]:
readings = [8.1, 0.0, None, 12.4]

print('if value         ->', [v for v in readings if v])
print('if v is not None ->', [v for v in readings if v is not None])

The first line silently dropped a valid reading of zero. Write `is not None` when you mean *is not None*.

## Four checks before you analyze anything

Well-formed JSON can still be wrong data. These are four separate questions, and passing one says nothing about the next:

| | Question | What failure looks like |
|---|---|---|
| 1. Decode | Can we read the bytes as text? | The wrong character encoding |
| 2. Parse | Does the text obey JSON grammar? | A trailing comma, an unmatched brace |
| 3. Validate | Does the record match our contract? | `"pm25": "12.4"` where a number belongs |
| 4. Interpret | Is the record fit for *this* analysis? | Duplicate station-hours, mixed units |

Most people stop after 2 and assume the rest. The last two are where wrong answers come from.

### Check 2 — does it parse?

Before running the next cell, find the four syntax errors below and name the rule each one breaks. Take a minute with the person next to you.

```
{
  'station_id': "A1",
  "active": True,
  "readings": [
    {"hour": 9, "pm25": .8},
  ]
}
```

The braces and brackets are all balanced, so do not count those. Indentation is not an error either.

In [ ]:
broken = '''
{
  'station_id': "A1",
  "active": True,
  "readings": [
    {"hour": 9, "pm25": .8},
  ]
}
'''

try:
    json.loads(broken)
except json.JSONDecodeError as err:
    print('message :', err.msg)
    print('line    :', err.lineno)
    print('column  :', err.colno)

The parser reports **the first problem it reaches and then stops**. One message does not mean one error — fix it, run again, and see what surfaces next.

The four faults: `'station_id'` uses single quotes; `True` should be `true`; `.8` needs a leading zero as `0.8`; and the readings array has a comma after its last item.

Repairing all four makes the document parse. It does **not** make the measurement correct, the units right, or the reading relevant to your study. That is what the next two checks are for.

One habit while you are here: never try to repair JSON with `eval` or a regular expression. If a real API response will not parse, print the first 200 characters first. Most of the time the server sent an HTML error page and your actual problem is the request.

### Check 3 — does it match the contract?

A **JSON Schema** is a written-down description of what a record must look like. It is itself written in JSON, which is a bit of a mind-bender the first time.

Here is a complete contract for one reading: both keys must be present, `hour` has to be a whole number from 0 to 23, `pm25` has to be a number or `null` and cannot be negative, and no other fields are allowed.

In [ ]:
try:
    from jsonschema import Draft202012Validator
    HAVE_SCHEMA = True
except ImportError:
    HAVE_SCHEMA = False
    print('jsonschema is missing. Run this once:  %pip install jsonschema')

reading_schema = {
    'type': 'object',
    'required': ['hour', 'pm25'],
    'properties': {
        'hour': {'type': 'integer', 'minimum': 0, 'maximum': 23},
        'pm25': {'type': ['number', 'null'], 'minimum': 0},
    },
    'additionalProperties': False,
}

if HAVE_SCHEMA:
    Draft202012Validator.check_schema(reading_schema)   # is the schema itself legal?
    validator = Draft202012Validator(reading_schema)
    print('schema is valid, validator ready')

Three things that schema buys you:

- **`required`** says which fields must exist. Listing a field under `properties` does *not* make it mandatory.
- **The type union** `['number', 'null']` lets `pm25` hold a real reading or an honest blank, while rejecting the string `"12.4"` — which is the failure that actually happens.
- **`additionalProperties: False`** rejects fields nobody declared. That is how you notice a typo in a key, or that the API added something in version 2.

Note `False` with a capital F. We are writing the schema as a Python dictionary, so it follows Python spelling. Written in a `.json` file it would be lowercase `false`.

Now run it over every reading in the document. The loop tracks where it is, so an error tells you which station and which reading, not just that something is wrong.

In [ ]:
def check_all(doc):
    problems = 0
    for i, station in enumerate(doc['stations']):
        for j, reading in enumerate(station['readings']):
            for err in validator.iter_errors(reading):
                where = ['stations', i, 'readings', j] + list(err.path)
                print(where, '->', err.message)
                problems += 1
    print('problems found:', problems)

if HAVE_SCHEMA:
    print('--- the document as loaded ---')
    check_all(data)

    print()
    print('--- now break one value on purpose ---')
    import copy
    tampered = copy.deepcopy(data)
    tampered['stations'][0]['readings'][0]['pm25'] = '8.1'   # a string, not a number
    check_all(tampered)

`"8.1"` looks like a number to a person and is a string to a computer. Nothing about it is malformed JSON — it parses perfectly. Only the contract catches it, and the path `['stations', 0, 'readings', 0, 'pm25']` walks you straight to the value in a document that might hold ten thousand of them.

That is the argument for schemas in one sentence. Parsing tells you the text is legal. A schema tells you the data is the shape you agreed on.

### The four Python operations, in one table

| Call | You give it | You get back |
|---|---|---|
| `json.loads(text)` | JSON in a string | Python values |
| `json.load(file)` | An open file | Python values |
| `json.dumps(obj)` | Python values | JSON in a string |
| `json.dump(obj, file)` | Python values and an open file | JSON written to disk |

The **s** means *string*. No `s` means *file*. `load` reads, `dump` writes.

One warning: `print(a_dict)` and `str(a_dict)` are **not** serialization. They give you Python's own formatting, complete with single quotes and `None`, which no JSON parser will accept.

## Traversing the readings

Before reaching for pandas, walk the structure by hand once. It is worth seeing what `json_normalize` is about to do for you.

In [ ]:
values = []
for station in data['stations']:
    for reading in station['readings']:
        value = reading['pm25']
        if value is not None:
            values.append(value)

print(values)

Three values, because one reading was `null`. Note the explicit `is not None` — same trap as before, and a station that genuinely measured `0.0` would be erased by `if value`.

There is a bigger problem with that list though: **it has lost the station identity.** You cannot answer "which station is highest" from three bare numbers. That is the gap pandas fills.

## Flatten it

Before typing anything, answer the question that decides every argument: **what is one row of my result?** Here it is one station at one hour, so four rows.

`json_normalize` needs two things to build that:

- `record_path` — which array supplies the rows. The readings.
- `meta` — which parent fields to copy down beside each row. The station ID and the site name, which live one and two levels up.

In [ ]:
import pandas as pd

df = pd.json_normalize(
    data['stations'],
    record_path='readings',
    meta=['station_id', ['site', 'name']],
    sep='.',
)

print('rows:', len(df))
df

Four rows, one per station-hour, with `station_id` and `site.name` repeated onto each one. The nested list `['site', 'name']` inside `meta` is how you reach a field that is itself inside an object.

The `null` came through as `NaN`, because the column is numeric and that is how pandas spells missing in a float column. Still missing, just wearing different clothes.

## Answer the question

In [ ]:
summary = (df.groupby('station_id')['pm25']
             .agg(mean_pm25='mean', n_observed='count')
             .round(1))
summary

**B2, at 11.6 µg/m³ against A1's 8.1.**

Now read the second column, because it matters more. `mean` skips missing values and `count` only counts the ones that were there, so A1's average rests on a single observation and B2's on two. Those are not comparable in any serious sense.

The number is correct and the claim would be irresponsible. Printing `n_observed` beside the mean is what lets somebody else see that for themselves.

## Write the result back out

Your output is somebody else's input, so send the context with it — the campaign and the units, not a bare table of numbers.

In [ ]:
result = {
    'campaign': data['campaign'],
    'units': data['units'],
    'summary': summary.reset_index().to_dict('records'),
}

with open('summary.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False, allow_nan=False)

print(Path('summary.json').read_text(encoding='utf-8'))

Three arguments doing real work:

- **`indent=2`** makes it readable by a person. Leave it off for machine-to-machine traffic where size matters.
- **`ensure_ascii=False`** writes `µg/m³` as those characters instead of escape sequences.
- **`allow_nan=False`** makes Python refuse to write `NaN` or `Infinity`. Neither is legal JSON, and Python emits them happily by default, producing a file that other languages choke on. Turning this on forces you to decide what a missing value should look like before it leaves your machine.

## Failures that hide inside valid JSON

All four of these parse without a complaint.

In [ ]:
print('1. repeated names, last one wins:')
print('  ', json.loads('{"pm25": 8.1, "pm25": 12.4}'))

print('2. Python accepts NaN even though JSON has no such thing:')
print('  ', json.loads('{"x": NaN}'))

print('3. a number that will not survive the trip:')
big = json.loads('{"id": 9007199254740993}')
print('   parsed by Python:', big['id'])
print('   through a float :', int(float(big['id'])), '<- last digit changed')

print('4. type drift, invisible until you do arithmetic:')
print('  ', json.loads('{"pm25": "8.1"}'))

Those are interoperability problems, which is a long word for *it worked on my machine*. Duplicate names are legal in the bare grammar and different parsers resolve them differently. `NaN` is outside the spec but Python both emits and accepts it. Integers past about nine quadrillion lose precision in any consumer that stores numbers as 64-bit floats, which includes JavaScript and therefore much of the web.

The defense is the same every time: unique names, declared types, IDs kept as strings, and `allow_nan=False` on the way out.

### Optional — pulling JSON from a live API

Wednesday's studio covers APIs properly. This is only to show that nothing changes once the bytes arrive.

In [ ]:
try:
    import requests
    resp = requests.get('https://api.github.com/repos/python/cpython', timeout=10)
    resp.raise_for_status()      # did the HTTP request succeed?
    repo = resp.json()           # can the body be decoded as JSON?
    print(repo['full_name'], '|', repo['stargazers_count'], 'stars')
except Exception as err:
    print('no network, or requests is missing:', type(err).__name__)
    print('skip it -- nothing later depends on this cell')

Two checks, two different failures. `raise_for_status` looks at the HTTP status code; `.json()` decodes the body. Either can fail on its own, and — the part people forget — **an error response often contains perfectly valid JSON.** A 404 from an API is usually a tidy little JSON object explaining that your thing does not exist. It parses. It is still not your data.

## Applied exercise — a third station

A station in the garden reported in late:

```json
{"station_id": "C3", "site": {"name": "Garden"},
 "readings": [
   {"hour": 9,  "pm25": 0.0},
   {"hour": 10, "pm25": null}
 ]}
```

**Write down your predictions before running anything:** how many rows will the flattened table have, how many observations will C3 contribute, and what will its mean be?

Then append it to the stations array, re-validate, re-flatten, and recompute the summary. Check your predictions against what you get.

*Extension:* change that `0.0` to the string `"0.0"` and run the schema check again. Which of the four checks catches it, and which ones let it through?

In [ ]:
# TODO: append the C3 station to data['stations']
# TODO: re-run the validation
# TODO: re-run json_normalize and the groupby
# TODO: compare against your three predictions

**My predictions, and what actually happened:** _..._

---

## One last question

Suppose a document parses cleanly and passes its schema.

**What could still make the mean you report wrong?**

Think about it before Wednesday. There are at least six good answers.